# Binary Classification Model

Task: Develop the best predictive model you can for (a) binary classification. You can use any preprocessing and any modeling technique you like, including ones not discussed in class (consider exploring RNN or SVM). Export a deployable copy of each model and a set of predictions from each model on the test data claims-test.RData. Predictions should be formatted as a single data frame named pred_df with columns

- .id containing the URL ID

- bclass.pred containing the predicted label for the binary class setting

- mclass.pred containing the predicted label for the multiclass setting


### RESULTS: 

<>

In [24]:
import pyreadr
import re
import pandas as pd
import numpy as np
import torch 
from torch import nn
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score, recall_score
import joblib
from sklearn.linear_model import LogisticRegression

## Load Data

In [3]:
raw_data_df = pyreadr.read_r('../../data/claims-raw.RData')["claims_raw"]
test_data_df = pyreadr.read_r('../../data/claims-test.RData')["claims_test"]


## Preprocess Claims Data 

Sourced from preprocessing.R

In [4]:
# function to parse html and clean text 
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "html.parser")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements
        if selector is None:
            raw = soup.get_text(' ')
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
    
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)
        text = re.sub("'", "",text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
        text = re.sub(r'nbsp', ' ', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text 
    
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None

# function to apply to test claims data
def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() 
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector)
    return df

In [5]:
clean_claims_df = parse_data(raw_data_df, selector = 'p')
test_clean_claims_df = parse_data(test_data_df, selector = 'p')

### Partition 

In [6]:
clean_claims_df.isna().sum()
clean_claims_df.head()

,neo_search_transaction_id,neo_search_subject_id,original_url,text_tmp,internal_feedback,.id,label,mclass,bclass,text_clean
0,12395162.0,11497914.0,http://hosting-22647.tributes.com/obituary/sho...,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 S...",Possible Fatality,url2183,Possible Fatality,Possible Fatality,Relevant claim content,national obituary search click on the item you...
1,12394582.0,11499214.0,https://www.localcrimenews.com/welcome/arrest/...,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",Potentially unlawful activity,url2136,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the following official arrest record for jonat...
2,12384260.0,11485861.0,https://www.bustedmugshots.com/florida/miami-d...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,N/A: No relevant content.,url1586,N/A: No relevant content.,N/A: No relevant content.,N/A: No relevant content.,did someone you know get arrested in miami dad...
3,12363093.0,11481705.0,https://www.policearrests.com/florida-arrest-r...,"<!DOCTYPE HTML>\n<html lang=""en"">\n<head>\n\t<...",Potentially unlawful activity,url209,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the information on this website is taken from ...
4,12376392.0,11488536.0,https://www.bustedmugshots.com/tennessee/memph...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,Potentially unlawful activity,url628,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,name clayton thomas location memphis tennessee...


In [7]:
data = clean_claims_df.dropna(subset=['text_clean'])
data = data[data['text_clean'].str.len() > 0] # drop rows with empty string 

targets, class_names = pd.factorize(data['bclass']) # convert labels to 0, 1 

X_train_text, X_val_text, y_train, y_val = train_test_split(data['text_clean'], targets, test_size = 0.2, stratify = targets, random_state = 2026)

### Vectorize

In [8]:
vectorizer = TfidfVectorizer(stop_words = 'english', min_df = 2, max_features = 5000)
X_train = vectorizer.fit_transform(X_train_text) # fit - learn vocab and IDF for training set 
X_val = vectorizer.transform(X_val_text) # transform only 
X_test = vectorizer.transform(test_clean_claims_df['text_clean'].fillna(''))
X_train.shape

(1575, 5000)

### Build Model 

In [9]:
class TextNN(nn.Module):
    def __init__(self, n_features, hidden=32, dropout=0.2):
        super().__init__()
        self.layer1 = nn.Linear(n_features, hidden)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.layer2 = nn.Linear(hidden, 1)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        return x.squeeze(-1)

### Convert to tensors 

In [10]:
X_train_t = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_val_t = torch.tensor(X_val.toarray(), dtype=torch.float32)
X_test_t = torch.tensor(X_test.toarray(), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

### Train Model

In [11]:
def train(hidden, dropout, lr, epochs = 10, batch_size = 32):
    torch.manual_seed(2026)
    
    model = TextNN(X_train_t.shape[1], hidden, dropout)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    loss_function = nn.BCEWithLogitsLoss()
    
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t),
        batch_size = batch_size, shuffle = True)
    
    for epoch in range(epochs):
        model.train()
            
        for x,y in loader: 
            optimizer.zero_grad()
            loss = loss_function(model(x), y)
            loss.backward()
            optimizer.step()
    
    return model

def predict_probs(model, x):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x)).numpy()
  

### Grid Search to find best parameters

In [12]:
param_grid = {'hidden': [16, 32, 64],
              'dropout': [0.3, 0.5],
              'lr': [0.0001, 0.001],
              'epochs': [5, 10]}

results = []

for params in ParameterGrid(param_grid):
    
    model = train(**params)
    
    auc = roc_auc_score(y_val, predict_probs(model, X_val_t))
    
    results.append({**params, 'val_auc': auc})
    
    print(params, f"VAL_AUC = {auc:.3f}")

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
results_df.head()

{'dropout': 0.3, 'epochs': 5, 'hidden': 16, 'lr': 0.0001} VAL_AUC = 0.896
{'dropout': 0.3, 'epochs': 5, 'hidden': 16, 'lr': 0.001} VAL_AUC = 0.911
{'dropout': 0.3, 'epochs': 5, 'hidden': 32, 'lr': 0.0001} VAL_AUC = 0.896
{'dropout': 0.3, 'epochs': 5, 'hidden': 32, 'lr': 0.001} VAL_AUC = 0.913
{'dropout': 0.3, 'epochs': 5, 'hidden': 64, 'lr': 0.0001} VAL_AUC = 0.903
{'dropout': 0.3, 'epochs': 5, 'hidden': 64, 'lr': 0.001} VAL_AUC = 0.918
{'dropout': 0.3, 'epochs': 10, 'hidden': 16, 'lr': 0.0001} VAL_AUC = 0.900
{'dropout': 0.3, 'epochs': 10, 'hidden': 16, 'lr': 0.001} VAL_AUC = 0.919
{'dropout': 0.3, 'epochs': 10, 'hidden': 32, 'lr': 0.0001} VAL_AUC = 0.903
{'dropout': 0.3, 'epochs': 10, 'hidden': 32, 'lr': 0.001} VAL_AUC = 0.921
{'dropout': 0.3, 'epochs': 10, 'hidden': 64, 'lr': 0.0001} VAL_AUC = 0.905
{'dropout': 0.3, 'epochs': 10, 'hidden': 64, 'lr': 0.001} VAL_AUC = 0.921
{'dropout': 0.5, 'epochs': 5, 'hidden': 16, 'lr': 0.0001} VAL_AUC = 0.896
{'dropout': 0.5, 'epochs': 5, 'hidden'

,dropout,epochs,hidden,lr,val_auc
23,0.5,10,64,0.001,0.921437
11,0.3,10,64,0.001,0.921255
9,0.3,10,32,0.001,0.920787
21,0.5,10,32,0.001,0.919696
7,0.3,10,16,0.001,0.918994


### Retrain Best Model

In [13]:
best = results_df.iloc[0]
print(best)
best_model = train(int(best['hidden']), best['dropout'], best['lr'], int(best['epochs']))

# check with validation to make sure its not voerfitting
train_probs = predict_probs(best_model, X_train_t)
train_preds = (train_probs > 0.5).astype(int)

print("train accuracy:", accuracy_score(y_train, train_preds))
print("train AUC:", roc_auc_score(y_train, train_probs))

val_probs = predict_probs(best_model, X_val_t)
val_preds = (val_probs > 0.5).astype(int)

print("val accuracy:", accuracy_score(y_val, val_preds))
print("val AUC:", roc_auc_score(y_val, val_probs))
print("sensitivity:", recall_score(y_val, val_preds, pos_label=0))
print("specificity:", recall_score(y_val, val_preds, pos_label=1))

print(confusion_matrix(y_val, val_preds))

dropout     0.500000
epochs     10.000000
hidden     64.000000
lr          0.001000
val_auc     0.921437
Name: 23, dtype: float64
train accuracy: 0.9403174603174603
train AUC: 0.9833323573331771
val accuracy: 0.8324873096446701
val AUC: 0.9214369234766793
sensitivity: 0.8790697674418605
specificity: 0.776536312849162
[[189  26]
 [ 40 139]]


### Predict with Test Set 

In [22]:
test_data_df.columns

test_probs = predict_probs(best_model, X_test_t)
test_clean_claims_df['pred_prob'] = test_probs
test_clean_claims_df['pred'] = class_names[(test_probs > 0.5).astype(int)]

test_clean_claims_df['pred'].value_counts()
test_clean_claims_df.head(10)

,.id,text_tmp,text_clean,pred_prob,pred
0,url3,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,name francine lewis location miami florida age...,0.137615,Relevant claim content
1,url6,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n<me...",raheem selby commited felony under federal jur...,0.413675,Relevant claim content
2,url8,<!DOCTYPE html>\r\n<html>\r\n<head>\r\n <me...,cjis data case details appearance details inca...,0.898781,N/A: No relevant content.
3,url9,"\n\n<script type=""text/javascript"">\n var t...",contact wigglesworth painting llc send me an e...,0.734258,N/A: No relevant content.
4,url11,"<!DOCTYPE HTML>\n<html lang=""en"">\n <head>\...",complaint against bill c spradlin david l pilv...,0.796790,N/A: No relevant content.
5,url16,<!DOCTYPE html>\r\n<html>\r\n<head>\r\n<meta n...,source toronto real estate board data january ...,0.993028,N/A: No relevant content.
6,url20,\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\t\r\n\r\n\t...,independently owned and operated phone fax ema...,0.945832,N/A: No relevant content.
7,url23,"<!DOCTYPE html><html lang=""en-US"" prefix=""og: ...",for businesses write a review filters suggeste...,0.923584,N/A: No relevant content.
9,url27,"\n<!DOCTYPE html>\n<html amp lang=""en"" id=""htm...",death has occurred or is near pre plan a funer...,0.070313,Relevant claim content
10,url30,"\n<!DOCTYPE html>\n<html lang=""en-US"">\n<head>...",a unique and lasting tribute for a loved one p...,0.014782,Relevant claim content


### Save work

In [ ]:
# import pickle 

# # save entire fitted model 
# with open("../../results/binary-model-preds.pkl", "wb") as file:
#     pickle.dump(best_model, file)

# # Create deployable copy of fitted model 
# # load entire model
# with open("../../results/binary-model-preds.pkl", "rb") as file:
#     loaded_model = pickle.load(file)


# save parameters
binary_parameters = {'state_dict': best_model.state_dict(),
                    'n_features': X_train_t.shape[1],
                    'hidden': int(best['hidden']),
                    'dropout': float(best['dropout']),
                    'class_names': [str(c) for c in class_names]}

torch.save(binary_parameters, "../../results/claims_binary_model.pt")

# load saved dict 
binary_parameters = torch.load("../../results/claims_binary_model.pt", map_location = "cpu", weights_only = True)

# reload model 
reload_model = TextNN(n_features = binary_parameters['n_features'],
                      hidden = binary_parameters['hidden'],
                      dropout =  binary_parameters['dropout'])

reload_model.load_state_dict(binary_parameters['state_dict'])
reload_model.eval()                             

# predict on test set 
test_pred_binary = predict_probs(reload_model, X_test_t)

pred_df = test_clean_claims_df[['.id']].copy()
pred_df['bclass.pred'] = np.array(binary_parameters['class_names'])[(test_pred_binary > 0.5).astype(int)]
pyreadr.write_rdata("../../results/binary-preds-emma.RData", pred_df, df_name = "pred_df")
#print(pred_df)


multi = pyreadr.read_r("deliverable3 preds multiclass.RData")
#print(multi)
print(multi.keys())

df1 = pred_df
df2 = multi['pred_df']

merged = pd.merge(df1, df2, on = ".id", how = "inner")
print(merged)

# Rdata file
pyreadr.write_rdata("../../results/preds-group3.RData", merged, df_name = "merged")

# csv file
merged.to_csv("../../results/preds-group3.csv", index = False)

odict_keys(['pred_df'])
         .id                bclass.pred                    mclass.pred
0       url3     Relevant claim content  Potentially unlawful activity
1       url6     Relevant claim content      N/A: No relevant content.
2       url8  N/A: No relevant content.      N/A: No relevant content.
3       url9  N/A: No relevant content.      N/A: No relevant content.
4      url11  N/A: No relevant content.      N/A: No relevant content.
..       ...                        ...                            ...
910  url3085  N/A: No relevant content.      N/A: No relevant content.
911  url3087  N/A: No relevant content.      N/A: No relevant content.
912  url3091     Relevant claim content  Potentially unlawful activity
913  url3093     Relevant claim content      N/A: No relevant content.
914  url3094     Relevant claim content              Possible Fatality

[915 rows x 3 columns]
